In [ ]:
#Installing Libraries

!pip -q install pandas numpy scikit-learn nltk matplotlib seaborn tqdm openai
!pip -q install transformers datasets accelerate torch --upgrade
print(" Libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 137.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 154.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 40.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.24.0+cu128 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.
torchaudio 2.9.0+cu128 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.
fastai 2.8.6 requires torch<2.10,>=1.10, but you have torch 2.10.0 which

In [ ]:

# Imports + NLTK

import os, re, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

nltk.download("punkt")
nltk.download("punkt_tab")
print("Setup complete.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


 Setup complete.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:

# Mounting Google Drive

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive
 Drive mounted.


In [ ]:
# Project Paths

PROJECT_DIR = Path("/content/drive/MyDrive/NLP_Research_Project")
DATA_DIR = PROJECT_DIR / "data"
OUT_DIR  = PROJECT_DIR / "outputs"

print("Project:", PROJECT_DIR)
print("Data   :", DATA_DIR)
print("Outputs:", OUT_DIR)

OUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f" Data folder not found: {DATA_DIR}\n Create it and put your .jsonl files there.")

print("\nFiles in DATA_DIR:")
for p in DATA_DIR.iterdir():
    print("-", p.name)
print("\n Paths ready.")

Project: /content/drive/MyDrive/NLP_Research_Project
Data   : /content/drive/MyDrive/NLP_Research_Project/data
Outputs: /content/drive/MyDrive/NLP_Research_Project/outputs

Files in DATA_DIR:
- news-snapchat-noagegroup-mental-2024.jsonl
- news-youtube-noagegroup-mental-2024.jsonl

 Paths ready.


In [ ]:
# Dataset Paths

snap_path = DATA_DIR / "news-snapchat-noagegroup-mental-2024.jsonl"
yt_path   = DATA_DIR / "news-youtube-noagegroup-mental-2024.jsonl"

print("Snap exists:", snap_path.exists(), "|", snap_path)
print("YT exists  :", yt_path.exists(),   "|", yt_path)

if not snap_path.exists() or not yt_path.exists():
    raise FileNotFoundError(" Missing dataset files in DATA_DIR. Check names + upload.")
print(" Dataset paths are good to go!")

Snap exists: True | /content/drive/MyDrive/NLP_Research_Project/data/news-snapchat-noagegroup-mental-2024.jsonl
YT exists  : True | /content/drive/MyDrive/NLP_Research_Project/data/news-youtube-noagegroup-mental-2024.jsonl
 Dataset paths OK.


In [ ]:
# OpenAI API Key Setup

from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Paste the lab OpenAI API key here: ")
print(" API key stored in environment.")

Paste the lab OpenAI API key here: ··········
 API key stored in environment.


In [ ]:
# Load JSONL Files

def load_jsonl_robust(path: Path, platform_name: str) -> pd.DataFrame:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                title = rec.get("title", "")
                body = rec.get("body") or rec.get("text") or ""
                if isinstance(body, list):
                    body = " ".join(body)
                full_text = f"{title}. {body}".strip()
                if len(full_text) < 20:
                    continue
                data.append({"platform": platform_name, "title": title, "full_text": full_text})
            except:
                continue
    return pd.DataFrame(data)

print("Loading datasets...")
df_snap = load_jsonl_robust(snap_path, "snapchat")
df_yt   = load_jsonl_robust(yt_path, "youtube")
df = pd.concat([df_snap, df_yt], ignore_index=True)

if df.empty:
    raise ValueError(" Loaded 0 articles. Check your jsonl format/files.")

# stable ID (prevents title-only collisions)
df["doc_id"] = df["platform"].astype(str) + "|" + df.index.astype(str) + "|" + df["title"].astype(str)

print(" Total loaded articles:", len(df))
display(df.head(3))

Loading datasets...
 Total loaded articles: 2002


,platform,title,full_text,doc_id
0,snapchat,Should Indonesia ban social media for children?,Should Indonesia ban social media for children...,snapchat|0|Should Indonesia ban social media f...
1,snapchat,Should Indonesia ban social media for children...,Should Indonesia ban social media for children...,snapchat|1|Should Indonesia ban social media f...
2,snapchat,Social Media's Grip on Teen Lives: A Deep Dive...,Social Media's Grip on Teen Lives: A Deep Dive...,snapchat|2|Social Media's Grip on Teen Lives: ...


In [ ]:
# LLM prompts and API client
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL_A = "gpt-4o"
MODEL_B = "gpt-4o-mini"
STANCE_LABELS = ["Positive", "Negative", "Neutral", "Mixed"]

RELEVANCE_PROMPT = """
You are a research annotator.

We study: News articles about YouTube/Snapchat (including Shorts/Spotlight) AND their impact on mental health / well-being.

IMPORTANT DEFINITIONS:
- "Impact" means the platform is described as influencing mental health outcomes (harm or benefit) via features, algorithms, addictive design, cyberbullying, body image effects, content exposure, policy, usage patterns, or interventions.
- NOT impact: the platform is only a hosting/distribution channel ("posted on YouTube") with no discussion of influence on mental health.

TASK:
Read the FULL article text. Estimate what percent (0–100) of the article is substantially about:
(1) mental health / well-being AND
(2) the platform's impact/influence on it.

Return ONLY valid JSON exactly like:
{
  "impact_relevance_percent": <0-100 integer>,
  "decision": "YES" or "NO",
  "reason": "<short reason, <= 15 words>"
}

Decision rule:
- YES if impact_relevance_percent >= 10
- NO otherwise
"""

STANCE_PROMPT = """
You are a stance annotator.

Task: stance of the article toward the platform’s impact on mental health.

Allowed labels ONLY: Positive, Negative, Neutral, Mixed

Return ONLY valid JSON exactly like:
{
  "label": "Positive/Negative/Neutral/Mixed",
  "reason": "<short reason, <= 15 words>"
}

Definitions:
- Positive: emphasizes benefits/support/resources/healthy usage mitigations
- Negative: emphasizes harms/risks/addiction/bullying/self-harm etc.
- Neutral: descriptive/unclear leaning about impact
- Mixed: strong positives and strong negatives both present
"""

def call_llm_json(model_name: str, prompt: str, article_text: str,
                 temperature: float = 0.0, max_tokens: int = 220) -> dict:
    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "Return ONLY valid JSON. No extra text."},
            {"role": "user", "content": prompt + "\n\nARTICLE:\n" + article_text}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    txt = resp.choices[0].message.content.strip()

    # parse JSON strictly; fallback to first {...}
    try:
        return json.loads(txt)
    except:
        m = re.search(r"\{.*\}", txt, flags=re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except:
                pass
    return {"_parse_error": True, "raw": txt}

def safe_call_json(model_name: str, prompt: str, article_text: str,
                   max_tokens: int = 220, max_retries: int = 3):
    for attempt in range(1, max_retries + 1):
        try:
            return call_llm_json(model_name, prompt, article_text, temperature=0.0, max_tokens=max_tokens)
        except Exception as e:
            wait = (2 ** attempt) + random.random()
            print(f" API error {model_name} attempt {attempt}/{max_retries}: {e}")
            time.sleep(wait)
    return {"_call_failed": True}

def get_pct(d):
    try:
        return int(d.get("impact_relevance_percent", -1))
    except:
        return -1

def get_dec(d):
    return str(d.get("decision", "UNK")).strip().upper()

def get_label(d):
    return str(d.get("label", "Unknown")).strip().title()

print("Ready for LLM client + % relevance + stance prompts.")

 Cell 8 ready: LLM client + % relevance + stance prompts.


In [ ]:
# Full LLM Run

OUTPUT_PARTIAL = OUT_DIR / "llm_results_partial.csv"
OUTPUT_FINAL   = OUT_DIR / "llm_results_final.csv"

for p in [OUTPUT_PARTIAL, OUTPUT_FINAL]:
    if p.exists():
        p.unlink()
print(" Removed old outputs. Starting fresh.")

SAVE_EVERY = 50
SLEEP_SECONDS = 0.12
MAX_CHARS = 7000

# Thresholds
DROP_PCT = 5
ACCEPT_PCT = 15

rows = []
print("Total articles to process:", len(df))

for _, r in tqdm(df.iterrows(), total=len(df)):
    doc_id = str(r["doc_id"])
    platform = str(r["platform"])
    title = str(r["title"])
    full_text = str(r["full_text"])

    article_text = f"PLATFORM: {platform}\nTITLE: {title}\nFULL_TEXT:\n{full_text[:MAX_CHARS]}"

    relA_raw = safe_call_json(MODEL_A, RELEVANCE_PROMPT, article_text, max_tokens=220)
    time.sleep(SLEEP_SECONDS)
    relB_raw = safe_call_json(MODEL_B, RELEVANCE_PROMPT, article_text, max_tokens=220)
    time.sleep(SLEEP_SECONDS)

    # Parse relevance
    pctA, pctB = get_pct(relA_raw), get_pct(relB_raw)
    decA, decB = get_dec(relA_raw), get_dec(relB_raw)

    # default values
    stanceA = stanceB = "NotRelevant"
    stanceA_raw = stanceB_raw = {}
    final_status = "REVIEW_RELEVANCE"
    final_label = "REVIEW"
    flag_reason = ""

    # handle parsing
    if relA_raw.get("_parse_error") or relB_raw.get("_parse_error") or relA_raw.get("_call_failed") or relB_raw.get("_call_failed"):
        final_status = "REVIEW_RELEVANCE"
        final_label = "REVIEW"
        flag_reason = "relevance_parse_or_call_failed"
    else:
        avg_pct = (pctA + pctB) / 2.0
        contradict = (decA == "YES" and decB == "NO") or (decA == "NO" and decB == "YES")

        if (avg_pct <= DROP_PCT) and not contradict:
            final_status = "DROP_NOT_MH"
            final_label = "DROP"
            flag_reason = f"avg_pct={avg_pct:.1f}"

        elif (avg_pct >= ACCEPT_PCT) and not contradict:
            stanceA_raw = safe_call_json(MODEL_A, STANCE_PROMPT, article_text, max_tokens=180)
            time.sleep(SLEEP_SECONDS)
            stanceB_raw = safe_call_json(MODEL_B, STANCE_PROMPT, article_text, max_tokens=180)
            time.sleep(SLEEP_SECONDS)

            stanceA = get_label(stanceA_raw)
            stanceB = get_label(stanceB_raw)

            if stanceA_raw.get("_parse_error") or stanceB_raw.get("_parse_error") or stanceA_raw.get("_call_failed") or stanceB_raw.get("_call_failed"):
                final_status = "REVIEW_STANCE"
                final_label = "REVIEW"
                flag_reason = "stance_parse_or_call_failed"
            else:
                if stanceA in STANCE_LABELS and stanceB in STANCE_LABELS and stanceA == stanceB:
                    final_status = "ACCEPT"
                    final_label = stanceA
                    flag_reason = f"avg_pct={avg_pct:.1f}"
                else:
                    final_status = "REVIEW_STANCE"
                    final_label = "REVIEW"
                    flag_reason = f"stanceA={stanceA}, stanceB={stanceB}, avg_pct={avg_pct:.1f}"
        else:
            final_status = "REVIEW_RELEVANCE"
            final_label = "REVIEW"
            flag_reason = f"avg_pct={(pctA+pctB)/2:.1f}, decA={decA}, decB={decB}"

    rows.append({
        "doc_id": doc_id,
        "platform": platform,
        "title": title,
        "full_text": full_text,

        "pct_A": pctA, "dec_A": decA, "rel_A_raw": json.dumps(relA_raw, ensure_ascii=False),
        "pct_B": pctB, "dec_B": decB, "rel_B_raw": json.dumps(relB_raw, ensure_ascii=False),

        "stance_A": stanceA, "stance_A_raw": json.dumps(stanceA_raw, ensure_ascii=False),
        "stance_B": stanceB, "stance_B_raw": json.dumps(stanceB_raw, ensure_ascii=False),

        "final_status": final_status,
        "final_label": final_label,
        "flag_reason": flag_reason
    })

    if len(rows) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(OUTPUT_PARTIAL, index=False)
        print(f" Checkpoint saved ({len(rows)} rows) -> {OUTPUT_PARTIAL}")

df_llm = pd.DataFrame(rows)
df_llm.to_csv(OUTPUT_FINAL, index=False)

print("\n Complete! :", OUTPUT_FINAL)
print(df_llm["final_status"].value_counts())

 Removed old outputs. Starting fresh.
Total articles to process: 2002


  2%|▏         | 50/2002 [04:22<2:51:15,  5.26s/it]

 Checkpoint saved (50 rows) -> /content/drive/MyDrive/NLP_Research_Project/outputs/llm_results_partial.csv


  5%|▍         | 100/2002 [08:24<2:39:46,  5.04s/it]

 Checkpoint saved (100 rows) -> /content/drive/MyDrive/NLP_Research_Project/outputs/llm_results_partial.csv


...

 Checkpoint saved (1950 rows) -> /content/drive/MyDrive/NLP_Research_Project/outputs/llm_results_partial.csv


100%|█████████▉| 2000/2002 [2:24:39<00:11,  5.65s/it]

 Checkpoint saved (2000 rows) -> /content/drive/MyDrive/NLP_Research_Project/outputs/llm_results_partial.csv


100%|██████████| 2002/2002 [2:24:50<00:00,  4.34s/it]



 FULL RUN COMPLETE: /content/drive/MyDrive/NLP_Research_Project/outputs/llm_results_final.csv
final_status
ACCEPT              932
REVIEW_RELEVANCE    463
REVIEW_STANCE       342
DROP_NOT_MH         265
Name: count, dtype: int64


In [ ]:
# Building Final Dataset + Manual Review Queue

df_llm = pd.read_csv(OUTPUT_FINAL)

df_accept = df_llm[df_llm["final_status"] == "ACCEPT"].copy()
df_review = df_llm[df_llm["final_status"].isin(["REVIEW_RELEVANCE", "REVIEW_STANCE"])].copy()

FINAL_LABELED_CSV = OUT_DIR / "final_labeled_dataset.csv"
MANUAL_REVIEW_CSV = OUT_DIR / "manual_review_queue.csv"

df_accept.to_csv(FINAL_LABELED_CSV, index=False)
df_review.to_csv(MANUAL_REVIEW_CSV, index=False)

print("Accepted labeled:", len(df_accept), "->", FINAL_LABELED_CSV)
print("Review queue    :", len(df_review), "->", MANUAL_REVIEW_CSV)
print(df_llm["final_status"].value_counts())

Accepted labeled: 932 -> /content/drive/MyDrive/NLP_Research_Project/outputs/final_labeled_dataset.csv
Review queue    : 805 -> /content/drive/MyDrive/NLP_Research_Project/outputs/manual_review_queue.csv
final_status
ACCEPT              932
REVIEW_RELEVANCE    463
REVIEW_STANCE       342
DROP_NOT_MH         265
Name: count, dtype: int64


In [ ]:
# Final Analysis + Case Studies

df_llm = pd.read_csv(OUTPUT_FINAL)
df_accept = pd.read_csv(FINAL_LABELED_CSV)
df_review = pd.read_csv(MANUAL_REVIEW_CSV)

print("Summary")
print("Total articles processed:", len(df_llm))
print("Accepted labeled:", len(df_accept))
print("Flagged for review:", len(df_review))

print("\n Final Status Breakdown:")
print(df_llm["final_status"].value_counts())

print("\n Platform-Wise Label Distribution")
platform_table = pd.crosstab(df_accept["platform"], df_accept["final_label"], margins=True)
print(platform_table)

print("\n Overall Label Distribution")
print(df_accept["final_label"].value_counts())

print("\n Case Studies (with examples)")
for i, row in df_review.head(5).iterrows():
    print("\n--- Case", i+1, "---")
    print("Platform:", row["platform"])
    print("Title:", row["title"])
    print("Final Status:", row["final_status"])
    print("pct_A/dec_A:", row["pct_A"], row["dec_A"])
    print("pct_B/dec_B:", row["pct_B"], row["dec_B"])
    print("stance_A:", row["stance_A"], "| stance_B:", row["stance_B"])
    print("Reason:", row["flag_reason"])
    print("Text snippet:", str(row["full_text"])[:350], "...")

===== SUMMARY =====
Total articles processed: 2002
Accepted labeled: 932
Flagged for review: 805

Final Status Breakdown:
final_status
ACCEPT              932
REVIEW_RELEVANCE    463
REVIEW_STANCE       342
DROP_NOT_MH         265
Name: count, dtype: int64

===== PLATFORM-WISE LABEL DISTRIBUTION (ACCEPTED ONLY) =====
final_label  Mixed  Negative  Neutral  Positive  All
platform                                            
snapchat        87       306       14         0  407
youtube         93       354        7        71  525
All            180       660       21        71  932

===== OVERALL LABEL DISTRIBUTION (ACCEPTED ONLY) =====
final_label
Negative    660
Mixed       180
Positive     71
Neutral      21
Name: count, dtype: int64

===== CASE STUDIES (FLAGGED EXAMPLES) =====

--- Case 1 ---
Platform: snapchat
Title: Social Media's Grip on Teen Lives: A Deep Dive | Headlines
Final Status: REVIEW_RELEVANCE
pct_A/dec_A: 5 NO
pct_B/dec_B: 20 YES
stance_A: NotRelevant | stance_B: NotReleva